# धडा १३ - एजंट स्मृती


## सेटअप

हा नोटबुक कसा तयार करायचा हे दर्शवतो **Microsoft Agent Framework** (MAF) वापरून **स्थायी मेमरी** असलेला प्रवास आरक्षण एजंट.

तुम्हाला वेगवेगळ्या प्रकारच्या एजंट मेमरी — वर्किंग, शॉर्ट-टर्म, आणि लॉंग-टर्म — एक एजंट संवादांदरम्यान माहिती कशी राखतो आणि वापरतो यावर कसा परिणाम होतो हे शिकवले जाईल.

**पूर्वापेक्षा गरजा:**  
- Azure AI Foundry प्रोजेक्ट ज्यात एक डिप्लॉय केलेला चॅट मॉडेल (उदा. `gpt-4o-mini`).  
- Azure CLI मध्ये लॉगिन केलेले — टर्मिनलमध्ये `az login` चालवा.  
- `AZURE_AI_PROJECT_ENDPOINT` — तुमच्या Azure AI Foundry प्रोजेक्टचा एन्डपॉइंट.  
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तुमच्या डिप्लॉय केलेल्या मॉडेलचे नाव.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजंट मेमरीचे प्रकार

एआय एजंट विविध प्रकारच्या मेमरीचा वापर करू शकतात, ज्यापैकी प्रत्येकाचा वेगळा उद्देश असतो:

### कार्यकारी मेमरी
संवाद धागा म्हणजेच एकाच सत्रातील देवाणघेवाण केलेली संदेशे. एजंट आधीच्या संदेशांकडे परत पाहू शकतो जेणेकरून संवाद सुसंगत राहील. MAF मध्ये हे **`agent.create_session()`** द्वारे तयार होते, जे `AgentSession` परत करते.

### अल्पकालिक मेमरी
तपशील जे कार्य किंवा सत्र संपेपर्यंत टिकून राहतात पण कायमस्वरूपी साठवले जात नाहीत. उदाहरणार्थ, एजंट बहु-चरण नियोजन संभाषणादरम्यान तथ्ये एकत्र करू शकतो आणि त्यांचा वापर अंतिम प्रवासात्मक योजनेसाठी करू शकतो.

### दीर्घकालिक मेमरी
ऐच्छिक आवडीनिवडी आणि तथ्ये जी **सत्रांमध्ये** टिकून राहतात. परत आलेल्या वापरकर्त्याला त्यांचे आहार प्रतिबंध किंवा प्रवासाची शैली वारंवार सांगावी लागत नाही. दीर्घकालिक मेमरी सहसा बाह्य संग्रहावर आधारित असते — डेटाबेस, फाइल किंवा व्हेक्टर निर्देशिका — आणि एजंटला साधनांद्वारे उपलब्ध केली जाते.


## सत्रांसह कार्य करणारे स्मृती

स्मृतीचा सर्वात सोपा प्रकार म्हणजे संभाषण सत्र. जेव्हा तुम्ही त्याच सत्राला (`agent.create_session()` द्वारे तयार केलेले) सलग `agent.run()` कॉल्सना पास करता, तेव्हा एजंट त्या संभाषणाचा पूर्ण इतिहास पाहू शकतो आणि पूर्वीच्या तपशीलांना आठवू शकतो.

चला एक प्रवास एजंट तयार करू आणि कार्य करणाऱ्या स्मृतीचे प्रदर्शन करू.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजंटने बजेट योग्यरित्या आठवले कारण दोन्ही संदेशांमध्ये एकाच सत्राची देवाणघेवाण झाली आहे. हेच **कार्य स्मृती** आहे — जी फक्त सत्राच्या आयुष्यासाठी अस्तित्वात असते.

### नवीन थ्रेडसह काय होतं?

जर आपण एक **नवीन** सत्र तयार केलं, तर एजंटला मागील संभाषणाची कोणतीही आठवण राहणार नाही:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालीन स्मृती नमुना

वापरकर्त्याच्या पसंती **सत्रांदरम्यान** लक्षात ठेवण्यासाठी, आपल्याला एक कायमस्वरूपी स्टोअर लागतो जो संभाषण थ्रेडच्या बाहेर असतो. एजंट या स्टोअरमध्ये प्रवेश करण्यासाठी **टूल्स** वापरतो — फंक्शन्स ज्यांना ते माहिती संग्रहित करण्यासाठी आणि पुनर्प्राप्तीसाठी कॉल करू शकते.

खाली आम्ही एक सोपा इन-मेमरी पसंती स्टोअर तयार करतो (उत्पादनात तुम्ही याला डेटाबेस किंवा वेक्टर इंडेक्सने बॅक करू शकता) आणि त्याला एजंट वापरू शकणाऱ्या टूल्स म्हणून उपलब्ध करून देतो.

### आर्किटेक्चर
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### परिदृश्य 1 — पहिल्यांदा वापरकर्ता वर्धापनदिनाच्या सफरीचे नियोजन करतो

सारा प्रथमच भेट देते. एजंटने तिच्या पसंती उपकरणांद्वारे संग्रहित कराव्या आणि त्यांचा वापर करून हॉटेलच्या शिफारशी कराव्यात.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### प्रकृती 2 — सारा आठवडे नंतर परतते

सारा एक **नवीन थ्रेड** सुरू करते (नवीन सत्राचे अनुकरण करत आहे). कार्यरत स्मृती रिकामी आहे, परंतु दीर्घकालीन पसंती संग्रहात तिची माहिती अजूनही आहे. एजंटने ती माहिती परत मागवून वैयक्तिक शिफारसींसाठी वापरावी.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

या धड्यात तुम्ही एजंट मेमरीचे तीन प्रकार आणि Microsoft Agent Framework वापरून त्यांना कसे अमलात आणायचे हे शिकलात:

| मेमरी प्रकार | MAF यंत्रणा | आयुष्यकाल |
|---|---|---|
| **काम करत असलेली** | `agent.create_session()` | एकदाच संभाषण |
| **अल्पकालीन** | थ्रेडमधील एकत्रित संदर्भ | एक कार्य / सत्र |
| **दीर्घकालीन** | `@tool` फंक्शन्सद्वारे प्रवेश केलेले बाह्य संग्रह | सत्रांमध्ये |

### मुख्य मुद्दे
1. **`agent.create_session()`** काम करत असलेली मेमरी देते — एजंट सत्रामध्ये पूर्ण संभाषणाचा इतिहास पाहतो.
2. **नवीन सत्र संदर्भ गमावते** — दीर्घकालीन मेमरीशिवाय एजंट मागील संभाषणे आठवू शकत नाही.
3. **`@tool` फंक्शन्स** भटकेपणा मिटवतात — ते एजंटला माहिती कायम ठेवण्यासाठी आणि पुन्हा मिळवण्यासाठी परवानगी देतात.
4. **वैयक्तिकरण वेळेनुसार सुधारते** — जितक्या जास्त प्राधान्यांचा संग्रह केला जातो, तितकी एजंटची शिफारस चांगली होते.

### वास्तविक जगाच्या वापरात
- **ग्राहक सेवा**: ग्राहकांची इतिहास आणि प्राधान्ये लक्षात ठेवणे
- **वैयक्तिक सहाय्यक**: दिवस किंवा आठवड्यांमध्ये संदर्भ राखणे
- **आरोग्यसेवा**: रुग्णांची माहिती आणि प्राधान्ये ट्रॅक करणे
- **ई-कॉमर्स**: इतिहासावर आधारित वैयक्तिकृत खरेदी

### पुढील पावले
- इन-मेमरी dict ऐवजी डेटाबेस किंवा व्हेक्टर स्टोअर वापरणे (उदा. Azure AI Search)
- वेळेनुसार माहितीच्या समाप्तीसाठी मेमरीची कालमर्यादा देणे
- सामायिक मेमरीसह मल्टि-एजंट सिस्टम तयार करणे
- नॉलेज-ग्राफ-आधारित मेमरीसाठी Cognee नोटबुकचा अभ्यास करणे


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI अनुवाद सेवेसह [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असूनसुद्धा, कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चूका किंवा असमर्थता असू शकते. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाच्या माहितींसाठी व्यावसायिक मानवी अनुवाद करण्याचा सल्ला दिला जातो. या अनुवादाच्या वापरातून उद्भवलेल्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थ लावण्याबद्दल आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
